In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("/kaggle/input/datasets/karvysingh/dalisa/FCC_feed_data.xlsx",
                   sheet_name="Feed Data")

# remove the units row
df = df[pd.to_numeric(df["Sample No."], errors="coerce").notna()].copy()

df["Sample No."] = df["Sample No."].astype(int)

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c])

df.head()

,Sample No.,Sp. Gr,CCR,T.S,T.N,IBP,5 %,10 %,30 %,50 %,70 %,90 %,95 %,FBP,P,N,A
1,1,0.9245,0.40,2.93,1401,37,317,342,392,423,454,498,514,538,8.3,24.1,67.6
2,2,0.9960,1.55,4.35,200,42,409,428,458,477,494,514,523,537,7.8,25.3,66.8
3,3,0.8683,0.04,0.03,22,103,224,263,337,380,420,473,497,536,15.2,45.0,39.8
4,4,0.9136,0.04,1.54,329,111,276,318,368,391,407,427,436,460,14.8,15.5,70.5
5,5,0.9303,0.28,3.06,1161,169,257,320,402,438,474,519,535,574,11.2,23.3,65.6


In [2]:
features = [
    "Sp. Gr",
    "5 %",
    "10 %",
    "30 %",
    "50 %",
    "70 %",
    "90 %",
    "95 %"
]

targets = ["P", "N", "A"]

train = df[df["Sample No."].between(1, 16)].copy()
test  = df[df["Sample No."].between(23, 28)].copy()
val   = df[df["Sample No."].between(17, 22)].copy()

X_train = train[features]
y_train = train[targets]

X_test = test[features]
y_test = test[targets]

X_val = val[features]
y_val = val[targets]

In [3]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
X_val   = scaler.transform(X_val)

In [4]:
import torch
import torch.nn as nn

from torch.func import functional_call
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import TensorDataset, DataLoader

class ANN(nn.Module):
    def __init__(self, h1, h2):
        super().__init__()

        self.fc1 = nn.Linear(8, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))       # tansig
        x = torch.sigmoid(self.fc2(x))    # logsig
        return self.out(x)                # purelin

In [5]:
DTYPE = torch.float64

Xtr = torch.tensor(X_train, dtype=DTYPE)
Xte = torch.tensor(X_test, dtype=DTYPE)
Xv  = torch.tensor(X_val, dtype=DTYPE)

In [6]:
!pip install -q torch-levenberg-marquardt
import torch_levenberg_marquardt as tlm

In [7]:
import copy
import io
import contextlib

def train_lm_early_stop(
    y_train,
    y_stop,
    h1,
    h2,
    seed=42,
    max_epochs=1000,
    max_fail=6
):
    torch.manual_seed(seed)

    model = ANN(h1, h2).double()

    ytr = torch.tensor(
        np.asarray(y_train),
        dtype=torch.float64
    ).reshape(-1, 1)

    ystop = torch.tensor(
        np.asarray(y_stop),
        dtype=torch.float64
    ).reshape(-1, 1)

    dataset = TensorDataset(Xtr, ytr)

    loader = DataLoader(
        dataset,
        batch_size=len(dataset),
        shuffle=False
    )

    lm = tlm.training.LevenbergMarquardtModule(
        model=model,
        loss_fn=tlm.loss.MSELoss(),
        learning_rate=1.0,
        attempts_per_step=10,
        solve_method="qr",
    )

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    failures = 0
    best_epoch = 0

    for epoch in range(max_epochs):

        # exactly one full-batch LM epoch
        # suppress package printing
        with contextlib.redirect_stdout(io.StringIO()):
            tlm.utils.fit(
                lm,
                loader,
                epochs=1
            )

        with torch.no_grad():
            stop_pred = model(Xte)

            stop_loss = torch.mean(
                (stop_pred - ystop) ** 2
            ).item()

        if stop_loss < best_loss:

            best_loss = stop_loss
            best_state = copy.deepcopy(
                model.state_dict()
            )

            best_epoch = epoch + 1
            failures = 0

        else:
            failures += 1

        if failures >= max_fail:
            break

    # Restore best generalizing checkpoint
    model.load_state_dict(best_state)

    print(
        f"best epoch = {best_epoch}, "
        f"stop RMSE = {np.sqrt(best_loss):.3f}"
    )

    return model

In [8]:
model_P = train_lm_early_stop(
    y_train["P"],
    y_test["P"],
    12, 13,
    seed=42
)

model_N = train_lm_early_stop(
    y_train["N"],
    y_test["N"],
    12, 12,
    seed=43
)

model_A = train_lm_early_stop(
    y_train["A"],
    y_test["A"],
    9, 9,
    seed=44
)

Epoch 1/1:   0%|          | 0/1 [00:00<?, ?it/s]

best epoch = 2, stop RMSE = 2.902
best epoch = 5, stop RMSE = 7.288


Epoch 1/1: 100%|██████████| 1/1 [00:00<00:00, 150.95it/s, epoch_avg_loss=7.3357e+01]

best epoch = 7, stop RMSE = 7.409


In [9]:
with torch.no_grad():

    P_raw = model_P(Xv).numpy().ravel()
    N_raw = model_N(Xv).numpy().ravel()
    A_raw = model_A(Xv).numpy().ravel()

total = P_raw + N_raw + A_raw

P = 100 * P_raw / total
N = 100 * N_raw / total
A = 100 * A_raw / total

pred = np.column_stack([P, N, A])

In [10]:
result = pd.DataFrame({
    "Sample": val["Sample No."].values,
    "P_exp": y_val["P"].values,
    "P_pred": P,
    "N_exp": y_val["N"].values,
    "N_pred": N,
    "A_exp": y_val["A"].values,
    "A_pred": A,
})

display(result.round(2))

,Sample,P_exp,P_pred,N_exp,N_pred,A_exp,A_pred
0,17,15.8,12.50,35.9,47.91,48.3,39.59
1,18,14.4,12.22,23.3,22.96,62.2,64.81
2,19,4.9,12.86,21.1,23.48,74.0,63.66
3,20,10.9,24.24,29.9,32.67,59.2,43.09
4,21,10.4,12.45,18.5,26.33,71.0,61.21
5,22,6.3,14.51,26.2,23.28,67.5,62.20


In [11]:
obs = y_val[["P", "N", "A"]].to_numpy().ravel()
prd = pred.ravel()

rmse = np.sqrt(
    np.mean((obs - prd)**2)
)

sse = np.sum((obs - prd)**2)
sst = np.sum((obs - obs.mean())**2)

r2 = 1 - sse / sst

print(f"RMSE = {rmse:.3f}")
print(f"R²   = {r2:.4f}")

RMSE = 7.920
R²   = 0.8841


In [12]:
def predict_pna(X):
    with torch.no_grad():
        p = model_P(X).numpy().ravel()
        n = model_N(X).numpy().ravel()
        a = model_A(X).numpy().ravel()

    s = p + n + a

    return np.column_stack([
        100*p/s,
        100*n/s,
        100*a/s
    ])

def evaluate(name, X, y):
    p = predict_pna(X)
    o = y[["P", "N", "A"]].to_numpy()

    rmse = np.sqrt(
        np.mean((o.ravel() - p.ravel())**2)
    )

    sse = np.sum((o.ravel() - p.ravel())**2)
    sst = np.sum(
        (o.ravel() - o.ravel().mean())**2
    )

    r2 = 1 - sse/sst

    print(
        f"{name:12s} RMSE = {rmse:.3f}   "
        f"R² = {r2:.4f}"
    )

evaluate("TRAIN", Xtr, y_train)
evaluate("TEST", Xte, y_test)
evaluate("VALIDATION", Xv, y_val)

TRAIN        RMSE = 7.287   R² = 0.8948
TEST         RMSE = 5.135   R² = 0.9322
VALIDATION   RMSE = 7.920   R² = 0.8841
